# Portfolio construction 2 of 3: how much

Notebooks: [1 hold](portfolio_construction_1_hold.ipynb) · [2 scale](portfolio_construction_2_scale.ipynb) · [3 trade](portfolio_construction_3_trade.ipynb), matching `src/portfolio_construction/{hold,scale,trade}`. In-sample, 1976-2014, DCC-GARCH risk model with the rescale (the configured model) and HRP-within-class hold sizing (the current default), primary cost 2 bps per traded notional (`primary_cost_bps`); data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`).

## Goal

Improve the model. Here: is the 10% risk ceiling the right control, and should scaling be a one-sided limit or a vol target?

## Current pipeline

**What we are doing.** Shrink the whole book when it uses too much leverage or risk. It never enlarges it.

**Inputs**
- Signed weights from the hold stage.
- Forecast covariance `Q` from the risk model.
- Config (`config/risk_config.json`): gross cap 3.0, risk ceiling 10%. I found no rationale for either in the repo.

**How we process them** (`scale/limits.py`)
1. `gross = sum of |weight|`.
2. `risk = sqrt(w'Qw)`, the forecast annual volatility of the book.
3. `scale = min(1, 3.0 / gross, 0.10 / risk)`.
4. Multiply every weight by `scale`.

*Example:* gross 4.5 and risk 12% give `min(1, 0.67, 0.83) = 0.67`. All positions shrink by a third, leaving gross 3.0 and risk 8.0%. The gross cap set the scale here, not the ceiling.

**Output.** The target book, passed to the trade stage.

## Implications

- The ceiling is a construction rule. The risk model supplies one number per month, `sqrt(w'Qw)`.
- Its strictness depends on risk-model calibration: under DCC-GARCH realised vol is about 2% above forecast, so the ceiling means what it says (the rolling model ran about 27% above).
- Being one-sided, it cannot lever up in calm months.

## Results and interpretation

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]


from risk_model_study import load_study, run, summarize
from run_pc_scale import model_parts

from src.backtest.qps_baseline import forecasts_from

pd.options.display.float_format = "{:.3f}".format
study = load_study()
settings, base, returns, source, known, meta = study.inputs
parts = model_parts(study, "configured")
targets, covariance, diagnostics = forecasts_from(parts, returns, meta, base)
ledgers = run(study, parts.risk_model)
print(f"{returns.shape[1]} markets; trend target months: {len(targets['trend'])}")

58 markets; trend target months: 468


In [2]:
bound = diagnostics.query("strategy == 'trend' and eligible >= @base['minimum_instruments']")
trend_ledger = ledgers["trend"]
pd.Series(
    {
        "months raw target breaches 3x gross cap": bound.target_gross_limited.mean(),
        "months raw target breaches 10% risk ceiling": bound.target_risk_limited.mean(),
        "mean target gross": bound.target_gross.mean(),
        "mean target forecast vol": bound.target_forecast.mean(),
        "calibration (realised / forecast vol)": summarize(trend_ledger)["calibration"],
    }
).to_frame("trend")

,trend
months raw target breaches 3x gross cap,0.700
months raw target breaches 10% risk ceiling,0.717
mean target gross,2.587
mean target forecast vol,0.083
calibration (realised / forecast vol),1.018


- **Limits bind almost always:** the raw target breaches the 3x gross cap in about 70% of months and the 10% ceiling in about 72%.
- **Calibration is 1.02:** realised vol (8.4%) is close to forecast (8.3%), so the ceiling is honest and edits can be compared at the same ceiling.

### 2.1 Which limit sets the scale?

Raw book from `construct_targets` with infinite limits.

In [3]:
from dataclasses import replace

from src.backtest.engine import simulate
from src.portfolio_construction import BandPolicy, PortfolioLimits
from src.risk_model import portfolio_risk
from src.transaction_cost_model import LinearBpsCost

raw_targets, raw_cov, _ = forecasts_from(
    replace(parts, limits=PortfolioLimits(np.inf, np.inf)), returns, meta, base
)
raw = raw_targets["trend"]
sample = returns.loc[base["start"] : base["end"]]
GROSS_CAP, CEILING = settings["gross_cap"], settings["portfolio_risk_ceiling"]

book = pd.DataFrame(
    {
        month: {
            "gross": np.abs(raw.loc[month]).sum(),
            "risk": portfolio_risk(raw.loc[month].to_numpy(), raw_cov[month]),
        }
        for month in raw.index
        if (raw.loc[month] != 0).sum() >= base["minimum_instruments"]
    }
).T
book["gross_ratio"] = GROSS_CAP / book.gross  # < 1 means the gross cap binds
book["risk_ratio"] = CEILING / book.risk  # < 1 means the risk ceiling binds
book["scale"] = book[["gross_ratio", "risk_ratio"]].clip(upper=1.0).min(axis=1)
gross_binds, risk_binds = book.gross_ratio < 1, book.risk_ratio < 1
which = np.select(
    [gross_binds & risk_binds, gross_binds, risk_binds],
    ["both", "gross cap only", "risk ceiling only"],
    default="neither",
)
by_case = book.groupby(which).agg(months=("scale", "size"), mean_scale=("scale", "mean"), mean_raw_risk=("risk", "mean"))
by_case["share"] = by_case.months / len(book)
any_binds = gross_binds | risk_binds
tightest = (book.risk_ratio < book.gross_ratio)[any_binds].mean()
print(f"Months where a limit binds: {any_binds.mean():.1%}; the risk ceiling is the tighter one in {tightest:.1%} of those")
print(f"Raw forecast vol: mean {book.risk.mean():.1%}, median {book.risk.median():.1%}, "
      f"months below the ceiling {(~risk_binds).mean():.1%}")
print(f"Months with room to scale UP to the ceiling (raw risk below it AND gross cap not binding): "
      f"{((~risk_binds) & (~gross_binds)).mean():.1%}")
by_case[["share", "months", "mean_scale", "mean_raw_risk"]]

Months where a limit binds: 98.3%; the risk ceiling is the tighter one in 41.2% of those
Raw forecast vol: mean 11.9%, median 11.3%, months below the ceiling 28.3%
Months with room to scale UP to the ceiling (raw risk below it AND gross cap not binding): 1.7%


,share,months,mean_scale,mean_raw_risk
both,0.435,203,0.679,0.124
gross cap only,0.266,124,0.725,0.090
neither,0.017,8,1.000,0.090
risk ceiling only,0.283,132,0.740,0.141


- A limit binds in **98%** of months: gross cap 70%, ceiling 72%, both 44%. Where one binds, the ceiling is the tighter in **41%**.
- Raw forecast vol averages 11.9% (median 11.3%), so the ceiling clips 72% of months. Mean forecast after limits: 8.3%.
- Only 1.7% of months have room to scale up under both limits. HRP's higher raw risk (the inverse-volatility book averaged 9.5%) leaves little to lever.

### 2.2 Limit versus vol target

Limit: `min(1, g/gross, c/risk)`. Target: `min(g/gross, c/risk)`, which also levers up. Same band, costs and executed limits. A third arm loosens the gross cap to 6x.

In [4]:
def evaluate(ceiling, lever_up, gross_cap=GROSS_CAP):
    scale = (gross_cap / book.gross).combine(ceiling / book.risk, min)
    if not lever_up:
        scale = scale.clip(upper=1.0)
    targets_ = raw.mul(scale.reindex(raw.index).fillna(0.0), axis=0)
    policy = BandPolicy(raw_cov, PortfolioLimits(gross_cap, ceiling), settings["primary_band"])
    ledger, _ = simulate(
        sample, targets_, LinearBpsCost(settings["primary_cost_bps"]),
        source.loc[sample.index], position_policy=policy, source_known_at=known.loc[sample.index],
    )
    row = summarize(ledger)
    row["share_levered_up"] = (scale > 1 + 1e-12).mean()
    row["mean_gross"] = ledger.gross_exposure.mean()
    return row

arms = {
    "limit (gross 3x)": {"lever_up": False},
    "target (gross 3x)": {"lever_up": True},
    "target (gross 6x)": {"lever_up": True, "gross_cap": 6.0},
}
grid = pd.concat(
    {
        (f"{c:.0%}", arm): evaluate(c, **options)
        for c in (0.06, 0.08, 0.10, 0.12, 0.14)
        for arm, options in arms.items()
    },
    axis=1,
).T
grid.index.names = ["ceiling / target", "policy"]
grid[["sharpe", "vol_ann", "calibration", "max_drawdown", "turnover_ann", "mean_cost_ann",
      "mean_forecast_vol", "mean_gross", "share_levered_up"]]

sharpe  vol_ann  calibration  \
ceiling / target policy                                            
6%               limit (gross 3x)    1.375    0.059        1.017   
                 target (gross 3x)   1.375    0.059        1.017   
                 target (gross 6x)   1.367    0.059        1.018   
8%               limit (gross 3x)    1.353    0.074        1.017   
                 target (gross 3x)   1.353    0.074        1.017   
                 target (gross 6x)   1.366    0.079        1.018   
10%              limit (gross 3x)    1.323    0.084        1.018   
                 target (gross 3x)   1.323    0.084        1.018   
                 target (gross 6x)   1.374    0.098        1.017   
12%              limit (gross 3x)    1.300    0.092        1.018   
                 target (gross 3x)   1.289    0.094        1.018   
                 target (gross 6x)   1.374    0.117        1.017   
14%              limit (gross 3x)    1.288    0.097        1.017   
                 target (gross 3x)   1.258    0.101        1.018   
                 target (gross 6x)   1.368    0.134        1.018   

                                    max_drawdown  turnover_ann  mean_cost_ann  \
ceiling / target policy                                                         
6%               limit (gross 3x)         -0.064        10.746          0.002   
                 target (gross 3x)        -0.064        10.746          0.002   
                 target (gross 6x)        -0.064        11.182          0.002   
8%               limit (gross 3x)         -0.085        12.775          0.003   
                 target (gross 3x)        -0.085        12.775          0.003   
                 target (gross 6x)        -0.085        14.877          0.003   
10%              limit (gross 3x)         -0.106        13.931          0.003   
                 target (gross 3x)        -0.106        13.955          0.003   
                 target (gross 6x)        -0.106        18.385          0.004   
12%              limit (gross 3x)         -0.119        14.640          0.003   
                 target (gross 3x)        -0.121        14.769          0.003   
                 target (gross 6x)        -0.126        21.421          0.004   
14%              limit (gross 3x)         -0.128        15.004          0.003   
                 target (gross 3x)        -0.141        15.388          0.003   
                 target (gross 6x)        -0.146        23.796          0.005   

                                    mean_forecast_vol  mean_gross  \
ceiling / target policy                                             
6%               limit (gross 3x)               0.057       1.930   
                 target (gross 3x)              0.057       1.930   
                 target (gross 6x)              0.058       1.980   
8%               limit (gross 3x)               0.072       2.321   
                 target (gross 3x)              0.072       2.321   
                 target (gross 6x)              0.078       2.638   
10%              limit (gross 3x)               0.081       2.523   
                 target (gross 3x)              0.081       2.526   
                 target (gross 6x)              0.097       3.277   
12%              limit (gross 3x)               0.087       2.639   
                 target (gross 3x)              0.088       2.656   
                 target (gross 6x)              0.115       3.852   
14%              limit (gross 3x)               0.090       2.692   
                 target (gross 3x)              0.093       2.746   
                 target (gross 6x)              0.131       4.316   

                                    share_levered_up  
ceiling / target policy                               
6%               limit (gross 3x)              0.000  
                 target (gross 3x)             0.000  
                 target (gross 6x)             0.002  
8%               limit (gross 3x)              0.000

- **Target at 3x gross is the limit** (Sharpe 1.323 for both at 10%): the gross cap blocks levering up.
- **Target at 6x gross, 10%:** levers up in 28% of months. Sharpe 1.374, vol 9.8%, turnover 18.4x (limit 13.9x), drawdown -10.6% (same as the limit). Its Sharpe is flat at 1.37 across ceilings, while the limit's falls from 1.375 (6%) to 1.288 (14%).
- **Nearest matched risk:** 6x target at 8% (vol 7.9%, Sharpe 1.366, turnover 14.9x) against the limit at 10% (vol 8.4%, Sharpe 1.323, turnover 13.9x): about +0.04 Sharpe for one more turn of turnover.
- **Calibration is 1.02 in every arm.**
- The gain is smaller than with inverse-volatility sizing (+0.05 against +0.08 at 10%), and turnover is much higher (13.9x for the limit against 8.7x). Single in-sample path; the Sharpe gaps are untested and the 15 variants add to the trial count.

## AFML improvements

AFML (Ch. 10, 14, 16) does not cover volatility targeting, leverage or gross caps. Two things carry over:

- **Deflated Sharpe (14.7.3):** the 15 variants above are trials.
- **Out-of-sample design (16.6):** compare scalings across many resampled paths, not one history.

## Suggested changes and experiments

| # | Experiment | Why |
|---|---|---|
| S1 | Paired Sharpe difference, limit against target at matched realised vol (`paired_sharpe_difference`) | The +0.08 is untested. |
| S2 | Sweep the gross cap (3x, 4x, 6x) under the target | Tests whether the gross cap is the real constraint. |
| S3 | Repeat on the configured model (DCC-GARCH with rescale) | Ceiling strictness depends on calibration. |
| S4 | Tail check before any copula or VaR/ES change: share of standardised returns below -2σ and -3σ against 2.3% and 0.13% | If tails are Gaussian and the std is above 1, fix the level (rescale). If tails are heavy, consider a tail measure in place of `sqrt(w'Qw)`. |

## Tests

Logic is in `scripts/run_pc_scale.py` (parameters in `config/pc_scale.json`); output goes to `reports/pc_scale/`, and this notebook runs the script if that folder is missing (a few minutes, mostly the DCC-GARCH fit). Each arm scales the raw (unlimited) TREND book each month by `min(g / gross, c / risk)`, clipped at 1 for a **limit** and unclipped for a **target**, then applies the same 10% band, executed limits and 2 bps cost as section 2.2. Arms: limit at gross 3x, targets at gross 3x, 4x and 6x. Ceilings: 6, 8, 10, 12, 14%. The same grid is run on two risk models: the rolling baseline and the configured DCC-GARCH with the rescale (`rescaled_components`).

**Reference** for every comparison: the limit at 3x gross and a 10% ceiling, which is the current pipeline.

**Assumptions I took.**
1. *The paired Sharpe difference uses 12-month blocks, 2,000 draws* (`paired_sharpe_difference`). Months are resampled; the book is not rebuilt.
2. *"Matched realised vol" means the ceiling whose realised volatility is nearest to the reference's.* The ceilings are on a coarse grid, so the match is approximate; the Sharpe difference is scale-free, so the mismatch moves risk, not the test.
3. *The gross-cap sweep runs under the target, at each ceiling,* with 3x, 4x and 6x.
4. *Trial count:* 20 arm-ceiling cells per model, 19 differences from the reference. 25 and 100 are sensitivities.
5. *No financing or leverage cost is charged beyond the 2 bps trading cost* (the project assumes zero cash income), so levering up is flattered.

In [5]:
from run_pc_scale import main

SCALE = ROOT / "reports/pc_scale_hrp"
if not SCALE.exists():
    main(SCALE)
grid = pd.read_csv(SCALE / "grid.csv")
paired = pd.read_csv(SCALE / "paired.csv")
tails = pd.read_csv(SCALE / "tails.csv")
deflated = pd.read_csv(SCALE / "deflated_sharpe.csv")
columns = ["sharpe", "vol_ann", "calibration", "max_drawdown", "turnover_ann", "mean_gross",
           "share_levered_up", "share_gross_binds"]
by_model = {m: g.set_index(["policy", "ceiling"])[columns] for m, g in grid.groupby("model")}
print(f"{len(grid)} cells; models: {sorted(by_model)}")

40 cells; models: ['configured', 'rolling']


### S1: is the limit-versus-target gap more than noise?

**What we are doing.** Test the +0.08 Sharpe from section 2.2 as a paired difference, so that the noise is that of the difference between two books built from the same markets and months.

**Inputs.** Monthly net returns of each target arm and the reference limit.

**Process.** Moving-block bootstrap of the shared months; report Sharpe(target) minus Sharpe(limit) with a 90% range, at the same ceiling and at the ceiling nearest in realised volatility. Then deflate each target-minus-reference return series for the number of cells tried.

**Output.** Difference, 90% range, deflated Sharpe.

In [6]:
paired[paired.model == "rolling"].drop(columns="model").set_index(["policy", "comparison"])

ceiling  vol_ann  difference  low_5pct  high_95pct
policy    comparison                                                      
target_3x same_ceiling    0.100    0.104       0.005    -0.006       0.015
          matched_vol     0.100    0.104       0.005    -0.006       0.015
target_4x same_ceiling    0.100    0.115       0.065     0.020       0.094
          matched_vol     0.080    0.099       0.099     0.030       0.146
target_6x same_ceiling    0.100    0.126       0.115     0.040       0.164
          matched_vol     0.080    0.101       0.121     0.043       0.169

- **On the rolling model with HRP sizing the 6x target beats the limit by +0.12 Sharpe, and the paired 90% range excludes zero:** +0.121 (0.043 to 0.169) at the ceiling nearest in volatility (8%: vol 10.1% against 10.2%), +0.115 at the same 10% ceiling. The 4x target at 8% gives +0.099 (0.030 to 0.146).
- **The 3x target adds nothing:** +0.005 (-0.006 to 0.015). The gross cap, not the policy name, decides the result.
- **The range is for this path only.** It resamples the months and keeps the book fixed, so it measures noise in the backtest, not selection. The configured model (S3) is the one that matters.

In [7]:
rolling_defl = deflated[deflated.model == "rolling"].set_index("arm_ceiling").drop(columns="model")
rolling_defl.loc[["target_3x_10%", "target_4x_8%", "target_4x_10%", "target_6x_8%", "target_6x_10%", "target_6x_12%"]]

,trials_19,trials_25,trials_100
arm_ceiling,,,
target_3x_10%,0.000,0.000,0.000
target_4x_8%,0.000,0.000,0.000
target_4x_10%,0.052,0.014,0.000
target_6x_8%,0.000,0.000,0.000
target_6x_10%,0.038,0.009,0.000
target_6x_12%,0.191,0.077,0.000


- **Deflation does not confirm it.** Counting the 19 cells tried, the deflated Sharpe of each target-minus-limit series is 0.00 to 0.05 for the cells shown (best of all: 6x at 14%, 0.203), and 0.01 or less at 25 trials. This is a hard test: the difference series is a levered book minus an unlevered one, so it holds extra exposure as well as the policy change. It says the gain does not stand out from the best of a family of similar leveraged variants; it does not say the gain is zero.

### S2: sweep the gross cap under the target

**What we are doing.** Test whether the 3x gross cap is the binding constraint, by raising it to 4x and 6x with everything else fixed.

**Output.** Sharpe, volatility, turnover, mean gross and the share of months that levered up and that the cap bound, by arm and ceiling (rolling model).

In [8]:
by_model["rolling"].loc[["limit_3x", "target_3x", "target_4x", "target_6x"]]

sharpe  vol_ann  calibration  max_drawdown  turnover_ann  \
policy    ceiling                                                             
limit_3x  0.060     1.205    0.074        1.304        -0.147         8.893   
          0.080     1.160    0.090        1.303        -0.196        10.063   
          0.100     1.106    0.102        1.304        -0.245        10.674   
          0.120     1.068    0.108        1.305        -0.284        10.865   
          0.140     1.068    0.109        1.305        -0.284        10.908   
target_3x 0.060     1.205    0.074        1.304        -0.147         8.893   
          0.080     1.160    0.090        1.303        -0.196        10.080   
          0.100     1.110    0.104        1.303        -0.245        10.893   
          0.120     1.059    0.116        1.304        -0.294        11.465   
          0.140     1.011    0.126        1.304        -0.343        11.890   
target_4x 0.060     1.223    0.076        1.304        -0.147         9.548   
          0.080     1.205    0.099        1.304        -0.196        11.849   
          0.100     1.171    0.115        1.303        -0.245        13.142   
          0.120     1.137    0.129        1.304        -0.294        14.045   
          0.140     1.096    0.142        1.304        -0.343        14.764   
target_6x 0.060     1.227    0.076        1.305        -0.147         9.633   
          0.080     1.227    0.101        1.305        -0.196        12.825   
          0.100     1.221    0.126        1.305        -0.245        15.610   
          0.120     1.206    0.148        1.305        -0.294        17.779   
          0.140     1.180    0.165        1.304        -0.343        19.244   

                   mean_gross  share_levered_up  share_gross_binds  
policy    ceiling                                                   
limit_3x  0.060         2.115             0.000              0.606  
          0.080         2.400             0.000              0.606  
          0.100         2.542             0.000              0.606  
          0.120         2.582             0.000              0.606  
          0.140         2.589             0.000              0.606  
target_3x 0.060         2.115             0.000              0.606  
          0.080         2.402             0.019              0.606  
          0.100         2.576             0.180              0.606  
          0.120         2.687             0.338              0.606  
          0.140         2.759             0.381              0.606  
target_4x 0.060         2.223             0.002              0.368  
          0.080         2.816             0.118              0.368  
          0.100         3.127             0.394              0.368  
          0.120         3.323             0.570              0.368  
          0.140         3.473             0.619              0.368  
target_6x 0.060         2.232             0.006              0.000  
          0.080         2.972             0.233              0.000  
          0.100         3.661             0.687              0.000  
          0.120         4.213             0.906              0.000  
          0.140         4.555             0.972              0.000

- **The gross cap is the real constraint (rolling model, HRP sizing).** At a 10% ceiling the Sharpe is 1.106 (limit), 1.110 (target 3x), 1.171 (4x), 1.221 (6x). The 3x cap binds in 61% of months (4x: 37%, 6x: 0%).
- **Levering up is what the gain needs:** the target levers up in 18% of months at 3x, 39% at 4x and 69% at 6x.
- **It costs risk and trading.** Volatility rises from 10.2% to 11.5% (4x) and 12.6% (6x), turnover from 10.7 to 13.1 and 15.6 times NAV, mean gross from 2.5 to 3.1 and 3.7.
- **About half the gain comes by 4x** (+0.065 of +0.115 at 10%); at an 8% ceiling the 4x target already scores 1.205 against 1.227 for 6x.
- **The target's Sharpe is flat in the ceiling, the limit's is not.** The 6x target scores 1.18 to 1.23 from 6% to 14%; the limit falls from 1.205 to 1.068.

### S3: the same grid on the configured model

**What we are doing.** Repeat S1 and S2 on DCC-GARCH with the rescale, where calibration is near 1, so the ceiling is closer to what it reads.

In [9]:
by_model["configured"].loc[["limit_3x", "target_3x", "target_4x", "target_6x"]]

sharpe  vol_ann  calibration  max_drawdown  turnover_ann  \
policy    ceiling                                                             
limit_3x  0.060     1.375    0.059        1.017        -0.064        10.746   
          0.080     1.353    0.074        1.017        -0.085        12.775   
          0.100     1.323    0.084        1.018        -0.106        13.931   
          0.120     1.300    0.092        1.018        -0.119        14.640   
          0.140     1.288    0.097        1.017        -0.128        15.004   
target_3x 0.060     1.375    0.059        1.017        -0.064        10.746   
          0.080     1.353    0.074        1.017        -0.085        12.775   
          0.100     1.323    0.084        1.018        -0.106        13.955   
          0.120     1.289    0.094        1.018        -0.121        14.769   
          0.140     1.258    0.101        1.018        -0.141        15.388   
target_4x 0.060     1.370    0.059        1.017        -0.064        11.127   
          0.080     1.375    0.078        1.017        -0.085        14.309   
          0.100     1.360    0.094        1.018        -0.106        16.503   
          0.120     1.341    0.106        1.018        -0.126        17.876   
          0.140     1.315    0.116        1.019        -0.146        18.902   
target_6x 0.060     1.367    0.059        1.018        -0.064        11.182   
          0.080     1.366    0.079        1.018        -0.085        14.877   
          0.100     1.374    0.098        1.017        -0.106        18.385   
          0.120     1.374    0.117        1.017        -0.126        21.421   
          0.140     1.368    0.134        1.018        -0.146        23.796   

                   mean_gross  share_levered_up  share_gross_binds  
policy    ceiling                                                   
limit_3x  0.060         1.930             0.000              0.700  
          0.080         2.321             0.000              0.700  
          0.100         2.523             0.000              0.700  
          0.120         2.639             0.000              0.700  
          0.140         2.692             0.000              0.700  
target_3x 0.060         1.930             0.000              0.700  
          0.080         2.321             0.000              0.700  
          0.100         2.526             0.017              0.700  
          0.120         2.656             0.103              0.700  
          0.140         2.746             0.191              0.700  
target_4x 0.060         1.976             0.000              0.413  
          0.080         2.572             0.019              0.413  
          0.100         2.997             0.137              0.413  
          0.120         3.246             0.317              0.413  
          0.140         3.416             0.445              0.413  
target_6x 0.060         1.980             0.002              0.026  
          0.080         2.638             0.032              0.026  
          0.100         3.277             0.278              0.026  
          0.120         3.852             0.593              0.026  
          0.140         4.316             0.788              0.026

In [10]:
paired[paired.model == "configured"].drop(columns="model").set_index(["policy", "comparison"])

ceiling  vol_ann  difference  low_5pct  high_95pct
policy    comparison                                                      
target_3x same_ceiling    0.100    0.084       0.001    -0.000       0.002
          matched_vol     0.100    0.084       0.001    -0.000       0.002
target_4x same_ceiling    0.100    0.094       0.038     0.001       0.059
          matched_vol     0.080    0.078       0.052     0.001       0.079
target_6x same_ceiling    0.100    0.098       0.051    -0.006       0.077
          matched_vol     0.080    0.079       0.044    -0.018       0.069

- **Calibration is 1.02 in every arm** (rolling: 1.30). A 10% ceiling gives 8.4% realised volatility in the limit, so the ceiling is tighter than it reads for risk appetite, but the forecast is right.
- **The level is higher before any scaling change:** limit Sharpe 1.323 with HRP sizing (inverse-volatility sizing gave 1.199), with a shallower drawdown (-10.6% against -11.6%) and 61% more turnover (13.9x against 8.7x).
- **The gap shrinks and loses significance:** target 6x minus limit at 10% is +0.051 (-0.006 to 0.077), so the range now includes zero. The 4x target is +0.038 (0.001 to 0.059) and 3x is +0.001. At matched volatility 6x is +0.044 (-0.018 to 0.069). The 4x target scores 1.360 against 1.374 for 6x, at mean gross 3.0 against 3.3 and turnover 16.5 against 18.4.
- **Deflation again is not met:** the best deflated difference is 0.005 (4x at 12%) at 19 trials, and 0.000 at 25 and 100.
- **The cost is again turnover:** 13.9 times NAV for the limit, 16.5 for the 4x target and 18.4 for 6x.
- **The 70% gross-cap binding rate** means most of the book is already at the cap, so there is little room for the target to add.

### S4: tail check before any copula or VaR/ES change

**What we are doing.** Test whether the forecast misses on level or on shape. A level miss is a standard deviation of standardised returns above 1; a tail miss is more mass below -2 and -3 standard deviations than a Gaussian has (2.3% and 0.13%) after the level is allowed for.

**Inputs.** Monthly gross return over the monthly volatility forecast (`standardized_returns`), 467 months, for the 10% limit and the 6x target.

In [11]:
shown = tails.set_index(["model", "policy"]).T
shown.loc["months below -3 sigma"] = (tails.set_index(["model", "policy"])["below_-3"] * tails.set_index(["model", "policy"])["months"]).round(0)
shown

model                   rolling           configured          
policy                 limit_3x target_6x   limit_3x target_6x
months                  467.000   467.000    467.000   467.000
std                       1.304     1.305      1.018     1.017
excess_kurtosis           5.824     5.786      0.481     0.465
below_-2                  0.017     0.017      0.013     0.013
below_-2_after_rescale    0.006     0.006      0.013     0.013
gaussian_-2               0.023     0.023      0.023     0.023
below_-3                  0.004     0.004      0.002     0.002
below_-3_after_rescale    0.002     0.002      0.002     0.002
gaussian_-3               0.001     0.001      0.001     0.001
months below -3 sigma     2.000     2.000      1.000     1.000

- **Rolling model (HRP sizing): a level miss and fat tails.** The standard deviation is 1.30 and excess kurtosis 5.8. Below -2 sigma is 1.7% and below -3 sigma 0.4% (about 2 months) against 0.13%.
- **Configured model: the level is fixed.** The standard deviation is 1.02, excess kurtosis 0.48, and the shares are 1.3% below -2 sigma (Gaussian 2.3%) and 0.2% below -3 sigma (about 1 month, against 0.13%).
- **Both arms give nearly identical numbers,** as expected: they scale the same book by the same forecast, and the standardised return does not depend on the scaling.
- **No sign of heavy tails on the configured model.** The rescale fixes the level. Keep `sqrt(w'Qw)` as the control.

## Summary and recommendations

**What we found** (TREND, HRP-within-class sizing, 10% band, 2 bps, 1976-2014, in sample; 20 cells per model).
- **Reference:** the configured model's 3x limit at 10% scores Sharpe 1.323, realised volatility 8.4%, calibration 1.02, turnover 13.9x.
- **S1:** on the rolling model the 6x target beats the limit by +0.12 Sharpe (paired 90% range 0.04 to 0.17). On the configured model the gain is +0.05 (-0.01 to 0.08) at the same ceiling, so the range includes zero for 6x. The 3x target adds nothing under either model. Deflated for 19 trials, nothing on the configured model clears 0.01.
- **S2:** the 3x gross cap binds in 61% to 70% of months and is the constraint. Raising it to 4x gives Sharpe 1.360 (configured, 10% ceiling) and 6x 1.374, against 1.323, at 18% to 32% more turnover and mean gross 3.0 to 3.3 against 2.5.
- **S3:** with HRP sizing the gain from levering up is smaller than it was with inverse-volatility sizing (+0.05 against +0.08), and the book is already near its limits (98% of months).
- **S4:** the configured model's level is fixed (standard deviation 1.02). Nothing justifies a tail-based risk measure.

**Recommendations**
1. **Keep the one-sided 3x limit as the production control.** The target at 3x gains nothing, and under the configured model the 6x target's gain is not distinguishable from zero.
2. **If a vol target is wanted, a 4x gross cap is the only candidate with a paired range above zero on the configured model** (+0.038, 0.001 to 0.059). It costs 18% more turnover than the limit and leaves out financing for 3.0x mean gross.
3. **Do not describe the gain as proven.** The deflated difference is at most 0.005 on the configured model, the window is already inspected, and the cells are correlated trials.
4. **Choose the ceiling by risk appetite, not Sharpe.** Under the 6x target the Sharpe is flat across 6% to 14% (1.37); the ceiling mainly sets volatility, drawdown and turnover.
5. **Keep the rescale.** It moves calibration from about 1.30 to 1.02.
6. **Turnover is now the main constraint.** HRP sizing has already raised turnover to 13.9x NAV at the 10% limit. A trade-stage change (notebook 3) or a higher cost level (5 bps) could reverse the HRP and vol-target gains. Rerun on a window not yet inspected before any adoption.

**Reference:** the earlier inverse-volatility results are in `reports/pc_scale/`.